# 12주차 실습 — 의료데이터에서 자주 생기는 오류

- **데이터**: `aact_mini` — ClinicalTrials.gov 임상시험 37,509건(2015~2024년 시작). 7~9주차와 같은 데이터입니다.
- **오늘 배우는 것**
  1. 건수가 부풀거나 줄어드는 원인을 찾아낼 수 있다
  2. 타입과 형식 때문에 생기는 오류를 알아볼 수 있다
  3. NULL과 표기 불일치를 확인하고 처리할 수 있다
  4. 점검 체크리스트로 결과를 스스로 검증할 수 있다
- **사용법**: 오류 6가지(1 · 2 · 3 · 4 · 9 · 10)마다 ① 동료의 SQL 실행 → ② 점검 SQL 실행 → ③ `___` 를 채워 고치고 `check(...)` 로 확인합니다(Q1~Q6). 동료의 SQL은 **오류 없이 실행되지만 답이 틀렸습니다.**
- **점수는 없습니다.** 오늘 공개하는 과제 2에서 같은 방법으로 배정 질환의 오류를 찾습니다.

## 0. 시작하기 전에

- **Colab 에서 열었다면 먼저 메뉴의 `파일 → Drive에 사본 저장`** 을 누르세요. 그래야 작성한 내용이 남습니다.
- 셀 실행: 셀 왼쪽의 ▶ 단추 또는 `Shift + Enter`. **위에서부터 차례로** 실행합니다.
- `%%sql` 로 시작하는 셀에는 SQL만 적습니다. `%%sql` 바로 아래의 `-- Q1` 같은 줄은 지우지 마세요 (문항 이름표입니다).
- 오류가 나면 붉은 글씨 중 **`Error:` 가 들어간 줄**만 읽으면 됩니다. 연결이 이상하면 `런타임 → 런타임 다시 시작` 후 준비 셀부터 다시 실행하세요.

- `___` 를 채우기 전에 실행하면 오류가 나는 것이 정상입니다.

In [ ]:
#@title ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게
%sql conn --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    _LAST.update(tag=tag, error=not result.success)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 학번이 입력되지 않아 배정 질환이 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

In [ ]:
#@title 채점용 정보 (수정하지 마세요) { display-mode: "form" }
ANSWERS = {'Q1': {'sig': 'cc0a92b11d2d', 'osig': 'cc0a92b11d2d', 'rows': 1, 'cols': ['시험수'], 'hint': '행이 아니라 시험 번호(nct_id)를 중복 없이 세는 COUNT 모양을 쓰세요 (8주차 COUNT 세 가지 모양).', 'ordered': False, 'optional': False}, 'Q2': {'sig': '366b4f6ad8d8', 'osig': '366b4f6ad8d8', 'rows': 1, 'cols': ['시험수'], 'hint': 'JOIN 뒤에는 한 시험이 여러 행입니다. 시험 단위로 중복 없이 세세요 (오류 1과 같은 방법).', 'ordered': False, 'optional': False}, 'Q3': {'sig': '5dff5c384615', 'osig': '5dff5c384615', 'rows': 1, 'cols': ['분모'], 'hint': '(INNER) JOIN 은 짝이 없는 시험을 뺍니다. 왼쪽 표(studies)의 행을 모두 남기는 JOIN 을 쓰세요 (8주차 JOIN 종류).', 'ordered': False, 'optional': False}, 'Q4': {'sig': '4f222f3ce6c3', 'osig': '4f222f3ce6c3', 'rows': 1, 'cols': ['count_star()'], 'hint': '점검 결과에서 타입이 숫자(BIGINT)인 열을 쓰세요. 숫자끼리는 크기로 비교됩니다.', 'ordered': False, 'optional': False}, 'Q5': {'sig': '6a71b87503a1', 'osig': '6a71b87503a1', 'rows': 1, 'cols': ['count_star()'], 'hint': '점검 결과에 보이는 값을 대소문자 · 띄어쓰기까지 그대로 쓰세요.', 'ordered': False, 'optional': False}, 'Q6': {'sig': 'dd701a919818', 'osig': 'dd701a919818', 'rows': 1, 'cols': ['count(distinct nct_id)'], 'hint': 'browse_conditions 표에서 MeSH 표준 용어가 원래 표기(대문자 포함)대로 들어 있는 열을 쓰세요 (슬라이드 327).', 'ordered': False, 'optional': False}, 'Q7': {'sig': '1533bdeb7270', 'osig': '1533bdeb7270', 'rows': 1, 'cols': ['전체', '값있음', '평균'], 'hint': '빈 값을 0으로 채우면 ‘아직 안 끝남’이 ‘0개월’이 됩니다. AVG 는 빈 값을 알아서 빼고 계산합니다.', 'ordered': False, 'optional': True}}

## 1. 세는 단위와 JOIN — 건수가 부풀거나 줄어든다

가장 흔하고, 가장 눈에 안 띄는 오류입니다.

### 오류 1 — 세는 단위를 잘못 잡음 (슬라이드 314)

**상황** “한국에서 진행된 임상시험이 몇 건인가요?”

In [ ]:
%%sql
-- [오류 예시] 동료의 SQL — 오류 없이 실행되지만 답이 틀렸습니다
SELECT COUNT(*) AS 시험수
FROM facilities
WHERE country = 'South Korea';

In [ ]:
%%sql
-- 점검 ① 한 행의 단위 확인 (슬라이드 336) — 행 수(첫 열)와 시험 수(둘째 열)가 같은가요?
SELECT COUNT(*), COUNT(DISTINCT nct_id)
FROM facilities;

**Q1.** 한 시험이 여러 기관에서 진행되어 기관 기록의 **행 수**를 센 것입니다. 시험 단위로 중복을 없애도록 고치세요.

In [ ]:
%%sql
-- Q1
-- 고친 SQL — ___ 를 채우세요
SELECT COUNT(___) AS 시험수
FROM facilities
WHERE country = 'South Korea';

In [ ]:
check("Q1", _)

> **병원 데이터라면** — 환자 수인지, 방문 건수인지, 처방 건수인지 먼저 정하세요. 표의 한 행이 무엇인지부터 확인합니다.

### 오류 2 — JOIN 뒤 행이 늘어남 (슬라이드 315)

**상황** 질환 정보를 붙였더니 건수가 10배가 되었습니다.

In [ ]:
%%sql
-- [오류 예시] 동료의 SQL — 오류 없이 실행되지만 답이 틀렸습니다
SELECT COUNT(*) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b
  ON s.nct_id = b.nct_id;

In [ ]:
%%sql
-- 점검 ① 한 행의 단위 확인 (슬라이드 336) — 붙이는 표(질환)는 한 시험에 몇 행인가요? (1 : N 관계)
SELECT COUNT(*), COUNT(DISTINCT nct_id)
FROM browse_conditions;

**Q2.** 한 시험에 질환이 평균 10개 붙어 행이 늘어났습니다. **시험 단위를 유지**하도록 고치세요. 두 표에 모두 있는 열은 `s.` 처럼 표 별칭을 붙여 씁니다. JOIN 전후의 행 수를 항상 비교하세요. 1 : N 이면 반드시 늘어납니다.

In [ ]:
%%sql
-- Q2
-- 고친 SQL — ___ 를 채우세요
SELECT COUNT(___) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b
  ON s.nct_id = b.nct_id;

In [ ]:
check("Q2", _)

### 오류 3 — INNER JOIN으로 분모가 줄어듦 (슬라이드 316)

**상황** “전체 시험 중 한국이 참여한 비율”을 구하려고 합니다. 먼저 분모(전체 시험 수)를 셉니다.

In [ ]:
%%sql
-- [오류 예시] 동료의 SQL — 오류 없이 실행되지만 답이 틀렸습니다
SELECT COUNT(DISTINCT s.nct_id) AS 분모
FROM studies AS s
JOIN facilities AS f
  ON s.nct_id = f.nct_id;

In [ ]:
%%sql
-- 점검 분모 — JOIN 하기 전 studies 의 전체 시험 수와 동료의 분모를 비교하세요
SELECT COUNT(*) AS 전체시험
FROM studies;

**Q3.** 기관 정보가 없는 2,699건이 조용히 빠졌습니다. **전체를 분모로 유지**하도록 JOIN 의 종류를 바꾸세요.

In [ ]:
%%sql
-- Q3
-- 고친 SQL — ___ 를 채우세요
SELECT COUNT(DISTINCT s.nct_id) AS 분모
FROM studies AS s
___ JOIN facilities AS f
  ON s.nct_id = f.nct_id;

In [ ]:
check("Q3", _)

## 2. 값의 타입과 형식 — 숫자처럼 보이지만 문자

### 오류 4 — 문자로 저장된 숫자 (슬라이드 319)

**상황** “최소 연령 60세 이상인 시험”을 찾으려고 합니다.

In [ ]:
%%sql
-- [오류 예시] 동료의 SQL — 오류 없이 실행되지만 답이 틀렸습니다
SELECT COUNT(*)
FROM eligibilities
WHERE minimum_age >= '60 Years';

In [ ]:
%%sql
-- 점검 값의 타입 확인 (슬라이드 335) — 슬라이드의 DESCRIBE 대신 7주차에 본 information_schema(표준 데이터 사전)로 봅니다
-- VARCHAR = 문자, BIGINT = 숫자
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE column_name LIKE 'minimum_age%';

**Q4.** 동료가 쓴 `minimum_age` 는 문자(VARCHAR)라서 **글자 순서로** 비교했습니다. '9 Years' 가 '60 Years' 보다 크게 나옵니다. **숫자형 열을 사용**하도록 고치세요.

`calculated_values` 는 AACT 가 계산해 둔 값(최소 연령을 숫자로 바꾼 `minimum_age_num` 등)을 모은 표입니다 (슬라이드 319).

In [ ]:
%%sql
-- Q4
-- 고친 SQL — ___ 를 채우세요
SELECT COUNT(*)
FROM calculated_values
WHERE ___ >= 60
  AND minimum_age_unit = 'year';

In [ ]:
check("Q4", _)

> **병원 데이터라면** — 혈압 `'120/80'`, 검사결과 `'<0.5'` 처럼 기호가 섞인 값은 모두 문자입니다.

## 3. 빈 값과 코드 — NULL · 대소문자 · 표기 불일치

### 오류 9 — 대소문자와 공백 (슬라이드 326)

**상황** 임상 단계를 'Phase 3' 으로 찾았습니다.

In [ ]:
%%sql
-- [오류 예시] 동료의 SQL — 오류 없이 실행되지만 답이 틀렸습니다
SELECT COUNT(*)
FROM studies
WHERE phase = 'Phase 3';

In [ ]:
%%sql
-- 점검 표기 불일치 — 값을 먼저 보고 조건 작성 (슬라이드 326)
SELECT DISTINCT phase
FROM studies;

**Q5.** 표기가 다르면 하나도 안 나옵니다. 점검 결과에서 3상의 **실제 값**을 찾아 따옴표 안의 `___` 에 그대로 쓰세요. 값이 0행이면 “없다”가 아니라 “조건이 틀렸다”를 먼저 의심하세요.

In [ ]:
%%sql
-- Q5
-- 고친 SQL — ___ 를 채우세요
SELECT COUNT(*)
FROM studies
WHERE phase = '___';

In [ ]:
check("Q5", _)

### 오류 10 — 같은 질환, 다른 표기 (슬라이드 327)

**상황** “류마티스 관절염” 시험을 연구자 입력 값(`conditions.name`)으로 찾았습니다.
(슬라이드에서는 제2형 당뇨병으로 보았습니다. 실습에서는 다른 질환으로 같은 오류를 확인합니다.)

In [ ]:
%%sql
-- [오류 예시] 동료의 SQL — 오류 없이 실행되지만 답이 틀렸습니다
SELECT COUNT(DISTINCT nct_id)
FROM conditions
WHERE name = 'Arthritis, Rheumatoid';

In [ ]:
%%sql
-- 점검 ② 값의 종류 확인 (슬라이드 336) — 같은 질환이 몇 가지로 적혀 있나요?
SELECT name, COUNT(*)
FROM conditions
WHERE name ILIKE '%rheumatoid%'
GROUP BY name
ORDER BY 2 DESC;   -- 2 = 두 번째 열

**Q6.** 자유 입력이라 표기가 여러 가지로 갈라져 있습니다. 표는 MeSH 표준 용어 표(`browse_conditions`)로 바꿔 두었습니다. **MeSH 표준 용어 열**(슬라이드 327)을 사용하도록 ___ 를 채우세요.

In [ ]:
%%sql
-- Q6
-- 고친 SQL — ___ 를 채우세요
SELECT COUNT(DISTINCT nct_id)
FROM browse_conditions
WHERE ___ = 'Arthritis, Rheumatoid';

In [ ]:
check("Q6", _)

> **병원 데이터라면** — 진단명 자유 입력 대신 KCD 코드를 쓰는 이유가 바로 이것입니다.

## 4. 시간 — 기준일 이후 정보가 섞인다

이 절의 오류 12와, 앞에서 실습하지 않은 오류 5 · 6 · 7 · 8 · 11 을 표 하나로 보고 넘어갑니다.

| 오류 (슬라이드) | 이렇게 하면 | 이렇게 고칩니다 |
|---|---|---|
| **오류 5** 문자로 저장된 날짜 (320) | `'2024-9-1' < '2024-10-1'` → false | `DATE '2024-09-01' < DATE '2024-10-01'` → true |
| **오류 6** 정수 나눗셈으로 0이 나옴 (321) | `191 / 817 * 100` → 0 | `ROUND(100.0 * 191 / 817, 1)` → 23.4 |
| **오류 7** 엑셀에서 앞자리 0이 사라짐 (322) | '0012345' → 12345 로 바뀌어 0행 | CSV를 열 때 그 열을 ‘텍스트’로 지정 |
| **오류 8** NULL을 0으로 바꿈 (324) | `AVG(COALESCE(검사값, 0))` → 실제보다 낮음 | `COUNT(*)` · `COUNT(검사값)` · `AVG(검사값)` 을 함께 봄 |
| **오류 11** 청구 자료를 진료 기록으로 읽음 (328) | 청구(EDI) 자료로 센 유병률이 실제보다 크게 나옴 | 세는 규칙을 문서로 남김 |
| **오류 12** 기준일 이후 정보가 섞임 (331~332) | 퇴원 후 검사까지 넣음 → 예측 정확도가 비현실적으로 높음 | `WHERE lab.검사일 <= adm.입원일` |

> 참고사항: 실습 DB(DuckDB)는 정수끼리 나눠도 소수로 계산해서 오류 6이 재현되지 않습니다 (슬라이드의 0은 PostgreSQL · SQL Server · SQLite). 어느 DB에서나 비율에는 `100.0` 을 곱하세요.

오류 8은 맨 아래 **더 해보기 (선택)** 에서 실제 데이터로 해 볼 수 있습니다.

## 5. 점검 체크리스트 — 오류를 찾는 SQL

결과를 내보내기 전에 이 표를 훑습니다 (슬라이드 335).

| 무엇을 | 어떻게 확인하나 | 이상하면 |
|---|---|---|
| 세는 단위 | `COUNT(*)` 와 `COUNT(DISTINCT 키)` 비교 | DISTINCT 를 쓰거나 먼저 집계 |
| JOIN 후 행 수 | JOIN 전후의 행 수 비교 | 1:N 관계 확인, 필요한 단위로 집계 |
| 분모 | LEFT JOIN 으로 다시 세기 | INNER JOIN 을 LEFT JOIN 으로 |
| 값의 타입 | DESCRIBE 로 타입 확인 (노트북: Q4 의 information_schema) | 숫자형 열을 쓰거나 CAST |
| 빈 값 | `COUNT(*)` 와 `COUNT(열)` 비교 | NULL 처리 규칙을 정하고 기록 |
| 표기 불일치 | DISTINCT 로 값의 종류 보기 | 표준 코드 열을 사용 |
| 이상값 | MIN · MAX · 상위 10건 보기 | 원인 확인 후 제외 기준 문서화 |
| 기준일 | 기준일 이후 자료가 섞였는지 | 날짜 조건을 다시 확인 |

**오류를 찾는 SQL 네 가지** (슬라이드 336) — 새 데이터를 받으면 가장 먼저 돌려 보는 조회입니다. 위에서 ①은 Q1 · Q2, ②는 Q6, ③은 더 해보기 Q7 에서 씁니다.

```sql
-- ① 한 행의 단위 확인
SELECT COUNT(*), COUNT(DISTINCT 키) FROM 표;
-- ② 값의 종류 확인
SELECT 열, COUNT(*) FROM 표 GROUP BY 열
ORDER BY 2 DESC;   -- 2 = 두 번째 열
-- ③ 빈 값 비율 확인
SELECT COUNT(*) AS 전체, COUNT(열) AS 값있음 FROM 표;
-- ④ 이상값 확인
SELECT MIN(열), MAX(열), AVG(열) FROM 표;
```

이 네 가지를 먼저 돌리면 대부분의 오류를 분석 시작 전에 발견합니다.

### 더 해보기 (선택)

**오류 8 — NULL을 0으로 바꿈 (슬라이드 324)** 을 실제 데이터로 해 봅니다.
시험의 실제 기간(개월, `calculated_values.actual_duration`)은 아직 끝나지 않았거나 끝난 날짜가 확인되지 않은 시험이면 비어 있습니다. 동료는 빈 값을 0으로 채워 평균을 냈습니다.

In [ ]:
%%sql
-- [오류 예시] 동료의 SQL — 오류 없이 실행되지만 답이 틀렸습니다
SELECT AVG(COALESCE(actual_duration, 0)) AS 평균
FROM calculated_values;

**Q7.** 슬라이드 324의 고친 SQL처럼 `전체` · `값있음`(점검 ③ 빈 값 비율)과 **빈 값을 뺀** `평균` 을 함께 내세요.

In [ ]:
%%sql
-- Q7
-- 고친 SQL — ___ 를 채우세요
SELECT COUNT(*) AS 전체,
       COUNT(actual_duration) AS 값있음,
       AVG(___) AS 평균
FROM calculated_values;

In [ ]:
check("Q7", _)

## 오늘 배운 것

- **세는 단위**와 **JOIN 후 행 수**를 항상 확인한다
- **타입 · 표기 · 빈 값**이 결과를 조용히 바꾼다
- **기준일 이전 자료만** 쓰고, 처리 방법을 보고서에 기록한다

과제 2(20점)는 오늘 공개되고, 13주차 수업 후 1주일 이내에 제출합니다.
다음 시간: 13강. CDW 실무 사례 — 코호트 추출을 처음부터 끝까지

아래 셀을 실행하면 오늘 확인한 문항(Q1~Q6, 선택 Q7)을 한눈에 볼 수 있습니다.

In [ ]:
진행확인()